# Investigation of the initial frames for the escape behavior videos

In [2]:
import cv2 as cv
import numpy as np
from pathlib import Path

In [ ]:
vid = Path(r"../data/FH0011 25-08-22 10-13-29.mkv")

In [6]:
img = cv.imread(str(vid), cv.IMREAD_GRAYSCALE)  # rename 'cap' -> 'img'

alpha = 1.0
beta  = 0.0

# Optional: show/save final result (keep an event loop for this window)
cont_applied = cv.convertScaleAbs(img, alpha=alpha, beta=beta)
def make_table_mask(gray: np.ndarray) -> np.ndarray:
    """Return a binary mask (uint8 0/255) for the round table in a grayscale frame."""
    # 1) Smooth a bit
    blur = cv.GaussianBlur(gray, (0,0), 3)

    # 2) Threshold (table is brightest). Otsu usually works well after you tuned alpha/beta.
    _, bin_ = cv.threshold(blur, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

    # (If lighting is uneven, try this instead):
    # bin_ = cv.adaptiveThreshold(blur, 255, cv.ADAPTIVE_THRESH_GAUSSIAN_C,
    #                             cv.THRESH_BINARY, 31, -10)

    # 3) Morphological cleanup
    k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (11,11))
    bin_ = cv.morphologyEx(bin_, cv.MORPH_CLOSE, k, iterations=2)
    bin_ = cv.morphologyEx(bin_, cv.MORPH_OPEN,  k, iterations=1)

    # 4) Keep the largest bright component (should be the table)
    num, labels, stats, _ = cv.connectedComponentsWithStats(bin_, connectivity=8)
    mask = np.zeros_like(bin_)
    if num > 1:
        largest = 1 + np.argmax(stats[1:, cv.CC_STAT_AREA])
        mask[labels == largest] = 255
    else:
        mask = bin_

    # 5) Optional: enforce circular shape (more stable for later analysis)
    cnts, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    if cnts:
        cnt = max(cnts, key=cv.contourArea)
        (cx, cy), r = cv.minEnclosingCircle(cnt)
        circ = np.zeros_like(mask)
        cv.circle(circ, (int(cx), int(cy)), int(r*0.98), 255, -1)
        mask = circ
       

    return mask

mask = make_table_mask(cont_applied)
table_only = cv.bitwise_and(cont_applied, cont_applied, mask=mask)
win = "Brightness"
cv.namedWindow(win, cv.WINDOW_NORMAL)

cv.createTrackbar("bright",   win, 100, 400, lambda *_: None)  # -100..300 after shift
cv.createTrackbar("contrast", win, 100, 500, lambda *_: None)  # 0..5.0 after /100

def draw():
    beta  = cv.getTrackbarPos("bright", win) - 100
    alpha = cv.getTrackbarPos("contrast", win) / 100.0
    out   = cv.convertScaleAbs(table_only, alpha=alpha, beta=beta)
    cv.imshow(win, out)

draw()
while True:
    if cv.waitKey(15) & 0xFF == 27:   # ESC
        break
    draw()

# Read FINAL slider values (don’t rely on callback return)
alpha = cv.getTrackbarPos("contrast", win) / 100.0
beta  = cv.getTrackbarPos("bright", win) - 100
print(f"alpha: {alpha}, beta: {beta}")
cv.destroyWindow(win)
def remove_inner_rectangles(gray, mask):
    edges = cv.Canny(gray, 60, 180)
    cnts, _ = cv.findContours(edges, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    H, W = mask.shape
    for c in cnts:
        area = cv.contourArea(c)
        if area < 0.005 * H * W:  # skip tiny stuff
            continue
        rect = cv.minAreaRect(c)                 # ((cx,cy),(w,h),angle)
        (cx, cy), (w, h), _ = rect
        if w == 0 or h == 0: 
            continue
        solidity = area / (w * h)                # ~1.0 for solid rectangles
        if solidity > 0.85 and mask[int(round(cy)), int(round(cx))] > 0:
            box = cv.boxPoints(rect).astype(np.int32)
            cv.fillPoly(mask, [box], 0)          # subtract rectangle
    return mask
table_mask = remove_inner_rectangles(table_only, mask)
tab = cv.namedWindow("Table Mask", cv.WINDOW_NORMAL)
cv.imshow("Table Mask", table_mask)
cv.waitKey(0)
cv.destroyAllWindows()

error: OpenCV(4.11.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\smooth.dispatch.cpp:618: error: (-215:Assertion failed) !_src.empty() in function 'cv::GaussianBlur'


In [ ]:
import cv2 as cv
import numpy as np
from pathlib import Path

# --- config ---
# vid can be a Path or string
vid = Path(r"\\172.25.250.112\burgalossi\lab share\Data\Florian\escape\FH7\FH0001 25-08-19 12-58-22.mkv")
out_path = "masked_output.avi"  # or None to skip saving

def make_table_mask(gray: np.ndarray) -> np.ndarray:
    """Return a binary mask (uint8 0/255) for the round table in a grayscale frame."""
    blur = cv.GaussianBlur(gray, (0, 0), 3)
    _, bin_ = cv.threshold(blur, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

    k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (11, 11))
    bin_ = cv.morphologyEx(bin_, cv.MORPH_CLOSE, k, iterations=2)
    bin_ = cv.morphologyEx(bin_, cv.MORPH_OPEN,  k, iterations=1)

    num, labels, stats, _ = cv.connectedComponentsWithStats(bin_, connectivity=8)
    mask = np.zeros_like(bin_)
    if num > 1:
        largest = 1 + np.argmax(stats[1:, cv.CC_STAT_AREA])
        mask[labels == largest] = 255
    else:
        mask = bin_

    cnts, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    if cnts:
        cnt = max(cnts, key=cv.contourArea)
        (cx, cy), r = cv.minEnclosingCircle(cnt)
        circ = np.zeros_like(mask)
        cv.circle(circ, (int(cx), int(cy)), int(r * 0.98), 255, -1)
        mask = circ
    return mask

def remove_inner_rectangles(gray: np.ndarray, mask: np.ndarray) -> np.ndarray:
    edges = cv.Canny(gray, 60, 180)
    cnts, _ = cv.findContours(edges, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    H, W = mask.shape
    for c in cnts:
        area = cv.contourArea(c)
        if area < 0.005 * H * W:
            continue
        rect = cv.minAreaRect(c)         # ((cx,cy),(w,h),angle)
        (cx, cy), (w, h), _ = rect
        if w == 0 or h == 0:
            continue
        solidity = area / (w * h)        # ~1.0 for solid rectangles
        if solidity > 0.85 and mask[int(round(cy)), int(round(cx))] > 0:
            box = cv.boxPoints(rect).astype(np.int32)
            cv.fillPoly(mask, [box], 0)  # subtract rectangle
    return mask

def process_video(vid,out_path):
    cap = cv.VideoCapture(str(vid))
    if not cap.isOpened():
        raise SystemExit(f"Could not open video: {vid}")

    # --- First frame: build the mask here ---
    ret, frame0 = cap.read()
    if not ret:
        raise SystemExit("Empty video / couldn't read first frame")

    if frame0.ndim == 3:
        gray0 = cv.cvtColor(frame0, cv.COLOR_BGR2GRAY)
    else:
        gray0 = frame0

    # Build initial mask on the first frame
    mask = make_table_mask(gray0)

    # Optional: remove overlay rectangles detected inside the table
    table_only0 = cv.bitwise_and(gray0, gray0, mask=mask)
    mask = remove_inner_rectangles(table_only0, mask)

    # (Optional) quick preview – press any key to continue
    # overlay = cv.cvtColor(gray0, cv.COLOR_GRAY2BGR)
    # cnts, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    # cv.drawContours(overlay, cnts, -1, (0, 255, 0), 2)
    # cv.imshow("First-frame mask (press any key)", overlay)
    # cv.waitKey(0)
    # cv.destroyAllWindows(); cv.waitKey(1)

    H, W = gray0.shape
    fps = cap.get(cv.CAP_PROP_FPS) or 30.0

    writer = None
    if out_path:
        fourcc = cv.VideoWriter_fourcc(*"MJPG")
        writer = cv.VideoWriter(out_path, fourcc, fps, (W, H), isColor=False)  # write BGR

    # --- Apply fixed mask to the REST of the video (including first frame if desired) ---
    #cv.namedWindow("Masked video", cv.WINDOW_NORMAL)

    # If you want to skip the first frame (since we already used it), uncomment:
    # pass
    # If you want to include it in output, process frame0 first:
    masked0 = cv.bitwise_and(gray0, gray0, mask=mask)
    vis0 = cv.cvtColor(masked0, cv.COLOR_GRAY2BGR)
    #cv.imshow("Masked video", vis0)
    if writer:
        writer.write(vis0)

    # Now process subsequent frames
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame.ndim == 3:
            gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
        else:
            gray = frame

        masked = cv.bitwise_and(gray, gray, mask=mask)  # apply the SAME mask
        #vis = cv.cvtColor(masked, cv.COLOR_GRAY2BGR)

        #cv.imshow("Masked video", vis)
        if writer:
            writer.write(masked)

        if cv.waitKey(1) & 0xFF == 27:  # ESC to stop early
            break

    cap.release()
    if writer:
        writer.release()
    cv.destroyAllWindows()

process_video(vid, out_path)
